# 02. Live Frame Histogram Change Detection Ablation

Sources: `notebooks/02-histogram-analysis.md`, `src/lib/cv/changeDetection.ts`, OpenCV `compareHist`, Swain & Ballard (1991).  
Goal: evaluate 4 histogram metrics x 3 color spaces x 3 stabilization windows and extract a `BEST_PARAMS` candidate.

Input: `data/live-frames/<scenario>/*.mp4` plus either `data/live-frames/<scenario>/ground-truth.csv` or root `data/live-frames/ground-truth.csv`.

In [ ]:
from pathlib import Path
import itertools
import json
import math
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
DOCS_DIR = ROOT / 'docs'
ABLATION_DIR = DOCS_DIR / 'ablation-results'
PIPELINE_DIR = DOCS_DIR / 'cv-pipeline'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Malgun Gothic')

print(f'ROOT={ROOT}')
print(f'OpenCV={cv2.__version__}')


def save_placeholder_png(path, title, message='No labeled data yet'):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.58, title, ha='center', va='center', fontsize=15, weight='bold')
    ax.text(0.5, 0.42, message, ha='center', va='center', fontsize=11)
    ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


from sklearn.metrics import auc, precision_recall_fscore_support, roc_curve

LIVE_DIR = DATA_DIR / 'live-frames'
SCENARIOS = ['normal-change', 'hand-shake', 'lighting', 'rolling-shutter', 'ios-autofocus']
METRICS = ['HISTCMP_CORREL', 'HISTCMP_CHISQR', 'HISTCMP_BHATTACHARYYA', 'HISTCMP_INTERSECT']
COLOR_SPACES = ['GRAY', 'HSV', 'RGB']
WINDOWS = [1, 3, 5]
METRIC_TO_CV = {
    'HISTCMP_CORREL': cv2.HISTCMP_CORREL,
    'HISTCMP_CHISQR': cv2.HISTCMP_CHISQR,
    'HISTCMP_BHATTACHARYYA': cv2.HISTCMP_BHATTACHARYYA,
    'HISTCMP_INTERSECT': cv2.HISTCMP_INTERSECT,
}

## Labels and Frame Extraction
Frames within +/-0.5 seconds of a labeled timestamp are treated as true scene changes.

In [ ]:
def scenario_label_path(scenario):
    nested = LIVE_DIR / scenario / 'ground-truth.csv'
    return nested if nested.exists() else LIVE_DIR / 'ground-truth.csv'

def load_labels():
    frames = []
    root_path = LIVE_DIR / 'ground-truth.csv'
    if root_path.exists():
        root = pd.read_csv(root_path)
        if 'scenario' not in root.columns:
            root['scenario'] = root['video_filename'].apply(lambda x: Path(str(x)).parent.name if '/' in str(x).replace('\\', '/') else 'unknown')
        frames.append(root)
    for sc in SCENARIOS:
        p = LIVE_DIR / sc / 'ground-truth.csv'
        if p.exists():
            df = pd.read_csv(p)
            df['scenario'] = sc
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=['video_filename', 'event_timestamp_sec', 'event_type', 'notes', 'scenario'])
    labels = pd.concat(frames, ignore_index=True).drop_duplicates()
    labels['event_timestamp_sec'] = pd.to_numeric(labels['event_timestamp_sec'], errors='coerce')
    return labels

labels = load_labels()
display(labels)

def find_videos():
    return sorted([p for ext in ('*.mp4', '*.mov', '*.avi', '*.mkv') for p in LIVE_DIR.glob(f'**/{ext}')])

videos = find_videos()
print(f'videos={len(videos)}')

In [ ]:
def extract_frames(video_path: Path, fps=10, max_frames=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'cannot open video: {video_path}')
    native_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    step = max(1, int(round(native_fps / fps)))
    out = []
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            ts = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
            out.append((ts, frame))
            if max_frames and len(out) >= max_frames:
                break
        idx += 1
    cap.release()
    return out

def video_events(video_path):
    name = video_path.name
    rel = str(video_path.relative_to(LIVE_DIR)).replace('\\', '/')
    hits = labels[(labels['video_filename'] == name) | (labels['video_filename'].astype(str).str.replace('\\', '/', regex=False) == rel)]
    return hits['event_timestamp_sec'].dropna().to_numpy(dtype=float)

def labels_for_timestamps(timestamps, events, tolerance=0.5):
    if len(events) == 0:
        return np.zeros(len(timestamps), dtype=int)
    return np.array([int(np.min(np.abs(events - t)) <= tolerance) for t in timestamps], dtype=int)

## Histogram Calculation and Change Scoring
CORREL/INTERSECT are converted so lower similarity means stronger change; CHISQR/BHATTACHARYYA already increase with change.

In [ ]:
def calc_hist(frame, color_space):
    if color_space == 'GRAY':
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    elif color_space == 'HSV':
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0], None, [180], [0, 180])
    elif color_space == 'RGB':
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        parts = [cv2.calcHist([rgb], [ch], None, [64], [0, 256]) for ch in range(3)]
        hist = np.concatenate(parts, axis=0)
    else:
        raise ValueError(color_space)
    return cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)

def similarity_scores(frames, metric, color_space):
    hists = [calc_hist(frame, color_space) for _, frame in frames]
    scores = [np.nan]
    for a, b in zip(hists[:-1], hists[1:]):
        scores.append(float(cv2.compareHist(a, b, METRIC_TO_CV[metric])))
    return np.array(scores, dtype=float)

def change_strength(scores, metric):
    if metric in ['HISTCMP_CORREL', 'HISTCMP_INTERSECT']:
        return 1 - np.nan_to_num(scores, nan=1.0)
    return np.nan_to_num(scores, nan=0.0)

def stabilize(pred, window):
    if window <= 1:
        return pred.astype(int)
    out = np.zeros_like(pred, dtype=int)
    run = 0
    for i, val in enumerate(pred):
        run = run + 1 if val else 0
        out[i] = int(run >= window)
    return out

def evaluate_scores(y_true, strength, window, threshold):
    pred = stabilize(strength >= threshold, window)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average='binary', zero_division=0)
    return {'precision': p, 'recall': r, 'f1': f1, 'tp': int(((pred == 1) & (y_true == 1)).sum()), 'fp': int(((pred == 1) & (y_true == 0)).sum()), 'fn': int(((pred == 0) & (y_true == 1)).sum()), 'tn': int(((pred == 0) & (y_true == 0)).sum())}

## 36-Combination Ablation and Heatmap
For each combination, the threshold with the best F1 score is selected from ROC candidate thresholds.

In [ ]:
records = []
roc_records = []
false_positive_images = []

for video in tqdm(videos, desc='videos'):
    scenario = video.parent.name
    frames = extract_frames(video, fps=10)
    if len(frames) < 2:
        continue
    timestamps = np.array([t for t, _ in frames])
    y_true = labels_for_timestamps(timestamps, video_events(video))
    for metric, color_space in itertools.product(METRICS, COLOR_SPACES):
        scores = similarity_scores(frames, metric, color_space)
        strength = change_strength(scores, metric)
        finite = np.isfinite(strength)
        if finite.sum() < 2 or len(np.unique(y_true[finite])) < 2:
            thresholds = np.linspace(np.nanmin(strength), np.nanmax(strength), 25) if finite.any() else [0.0]
        else:
            fpr, tpr, thr = roc_curve(y_true[finite], strength[finite])
            roc_records.append({'scenario': scenario, 'metric': metric, 'color_space': color_space, 'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': float(auc(fpr, tpr))})
            thresholds = thr[np.isfinite(thr)]
        if len(thresholds) == 0:
            thresholds = [0.0]
        for window in WINDOWS:
            evals = []
            for threshold in thresholds:
                m = evaluate_scores(y_true, strength, window, float(threshold))
                evals.append({**m, 'threshold': float(threshold)})
            best = max(evals, key=lambda x: x['f1'])
            records.append({'video': str(video.relative_to(LIVE_DIR)), 'scenario': scenario, 'metric': metric, 'color_space': color_space, 'window': window, **best})

results = pd.DataFrame(records)
if results.empty:
    print('No video/label data found; saving an empty result template.')
    results = pd.DataFrame(columns=['video', 'scenario', 'metric', 'color_space', 'window', 'threshold', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn', 'tn'])
results.to_csv(ABLATION_DIR / 'histogram-ablation.csv', index=False)
display(results.head())

summary = results.groupby(['metric', 'color_space', 'window'], dropna=False).agg(precision=('precision', 'mean'), recall=('recall', 'mean'), f1=('f1', 'mean'), threshold=('threshold', 'median')).reset_index() if not results.empty else pd.DataFrame()
if not summary.empty:
    display(summary.sort_values('f1', ascending=False).head(10))
    heat = summary.pivot_table(index='metric', columns=['color_space', 'window'], values='f1')
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(heat, annot=True, fmt='.2f', cmap='mako', ax=ax)
    ax.set_title('Histogram change detection F1')
    fig.tight_layout()
    fig.savefig(ABLATION_DIR / 'histogram-heatmap.png', dpi=180)
    plt.show()
else:
    save_placeholder_png(ABLATION_DIR / 'histogram-heatmap.png', 'Histogram heatmap')

## ROC Curves and False Positive Gallery
Scenario ROC plots are saved to `histogram-roc-per-scenario.png` and the handoff-compatible `histogram-roc.png`.

In [ ]:
if roc_records:
    best_combo = summary.sort_values('f1', ascending=False).iloc[0]
    fig, ax = plt.subplots(figsize=(7, 6))
    for rec in roc_records:
        if rec['metric'] == best_combo.metric and rec['color_space'] == best_combo.color_space:
            ax.plot(rec['fpr'], rec['tpr'], label=f"{rec['scenario']} AUC={rec['auc']:.2f}")
    ax.plot([0, 1], [0, 1], '--', color='gray')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend()
    fig.tight_layout()
    fig.savefig(ABLATION_DIR / 'histogram-roc-per-scenario.png', dpi=180)
    fig.savefig(ABLATION_DIR / 'histogram-roc.png', dpi=180)
    plt.show()
else:
    print('ROC requires both positive and negative labels.')
    save_placeholder_png(ABLATION_DIR / 'histogram-roc-per-scenario.png', 'Histogram ROC per scenario')
    save_placeholder_png(ABLATION_DIR / 'histogram-roc.png', 'Histogram ROC')

save_placeholder_png(PIPELINE_DIR / 'histogram-false-positives.png', 'Histogram false positives')
print('False positive gallery can be saved to docs/cv-pipeline/histogram-false-positives.png after labeled data is available.')

## BEST_PARAMS Candidate
The printed JSON is the evidence for updating `src/lib/cv/changeDetection.ts`. If no labeled data exists, keep the current default.

In [ ]:
if not summary.empty:
    best = summary.sort_values('f1', ascending=False).iloc[0]
    # TS threshold is similarity-based. Convert the selected strength threshold back to similarity threshold.
    if best.metric in ['HISTCMP_CORREL', 'HISTCMP_INTERSECT']:
        ts_threshold = 1 - float(best.threshold)
    else:
        ts_threshold = 1 - float(best.threshold)
    recommendation = {
        'metric': str(best.metric),
        'colorSpace': str(best.color_space),
        'windowSize': int(best.window),
        'threshold': round(ts_threshold, 4),
        'meanF1': round(float(best.f1), 4),
    }
else:
    recommendation = {'metric': 'HISTCMP_CORREL', 'colorSpace': 'GRAY', 'windowSize': 3, 'threshold': 0.92, 'meanF1': None}
print(json.dumps(recommendation, ensure_ascii=False, indent=2))